# 13 · Model misspecification — worked solutions

These solutions are most useful after attempting the chapter exercises. Each
section states the reasoning before the calculation; numerical values depend on
the compact, fixed-seed setup below.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import geodef

rng = np.random.default_rng(0)
fault = geodef.Fault.planar(
    lat=34.0, lon=-118.0, depth=9_000.0, strike=90.0, dip=30.0,
    length=24_000.0, width=12_000.0, n_length=4, n_width=3,
)
i = np.arange(fault.n_patches) % 4
j = np.arange(fault.n_patches) // 4
true_slip = np.exp(-((i - 1.5) / 1.2) ** 2 - ((j - 1.0) / 0.9) ** 2)
station_lon, station_lat = np.meshgrid(
    np.linspace(-118.18, -117.82, 6), np.linspace(33.90, 34.12, 5)
)
station_lon, station_lat = station_lon.ravel(), station_lat.ravel()
east, north, up = fault.displacement(
    station_lat, station_lon, slip_strike=0.0, slip_dip=true_slip
)
sigma = 0.004
gnss = geodef.data.gnss(
    lon=station_lon, lat=station_lat,
    east=east + rng.normal(0.0, sigma, east.size),
    north=north + rng.normal(0.0, sigma, north.size),
    up=up + rng.normal(0.0, sigma, up.size),
    sigma_east=sigma, sigma_north=sigma, sigma_up=sigma,
    name="solution_gnss",
)


## 1. Dip sensitivity

Generate once with the true dip, then solve every trial with identical data and
regularization. Plot bias against dip rather than selecting only two cases.

## 2–3. Noise and discretization

White-noise inversion of correlated data underestimates uncertainty. A coarse
mesh aliases short-wavelength slip into the nearest representable pattern.

## 4. Station design

Depth errors are exposed where competing geometries predict the largest
normalized difference, subject to real siting constraints.


In [ ]:
dip_bias = {}
true_mw = fault.magnitude(true_slip)
for dip in (20.0, 30.0, 40.0, 50.0):
    trial = geodef.Fault.planar(
        lat=34.0, lon=-118.0, depth=9_000.0, strike=90.0, dip=dip,
        length=24_000.0, width=12_000.0, n_length=4, n_width=3,
    )
    fit = geodef.solve(
        trial, datasets=gnss, components="dip", regularization="laplacian",
        regularization_strength=1.0, bounds=(0.0, None),
    )
    dip_bias[dip] = trial.magnitude(fit.dip_slip) - true_mw
print("Mw bias by assumed dip:", dip_bias)


## Interpretation and alternatives

The most dangerous misspecification may have modest residuals because flexible slip absorbs it; sensitivity of derived quantities remains visible.
